In [ ]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
from datetime import datetime
import joblib
from data import DataReader, NumpyDataset, HybridNormalizer
from sklearn.preprocessing import RobustScaler

time_pairs = [('2024030105', '2024030106'), ('2024030117', '2024030118')]

directories = {
    'pm25': 'your_path/pm25',
    'output': 'your_path/output',
    'T2': 'your_path/T2',
    'E_pm25': 'your_path/E_pm25',
    'HGT': 'your_path/HGT',
    'U10': 'your_path/U10',
    'V10': 'your_path/V10',
    'LAI': 'your_path/LAI',
    'QVAPOR': 'your_path/QVAPOR',
    'PBLH': 'your_path/PBLH',
    'UST': 'your_path/UST',
    'HFX': 'your_path/HFX',
}

val_data_reader = DataReader(
    i_range=range(1, 8),
    j_range=range(1, 12),
    directories=directories,
    time_pairs=time_pairs,
)

val_normalizer = HybridNormalizer()
val_normalizer.load("./hybrid_normalizer.pkl")

val_dataset = NumpyDataset(
    data_reader=val_data_reader,
    normalizer=val_normalizer,
    transform=None,
)

val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [ ]:
import torch
import numpy as np
import os
from FNO import FNO2d

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

output_directory = 'your_output_path/result'

time_pairs = [('2024030105', '2024030106'), ('2024030117', '2024030118')]

model = FNO2d(in_channels=len(directories)-1, out_channels=1,
              modes1=20, modes2=24, width=32).to(device)
checkpoint = torch.load('./best_fno_model.pth', map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

val_normalizer = HybridNormalizer()
val_normalizer.load("./hybrid_normalizer.pkl")

print("Model and normalizer loaded, starting time-loop prediction...")

for t_idx, (input_time, target_time) in enumerate(time_pairs):
    print(f"\n[{t_idx+1}/{len(time_pairs)}] Processing target time: {target_time} (input: {input_time})")

    current_reader = DataReader(
        i_range=range(1, 8),
        j_range=range(1, 12),
        directories=directories,
        time_pairs=[(input_time, target_time)]
    )

    current_dataset = NumpyDataset(
        data_reader=current_reader,
        normalizer=val_normalizer,
        transform=None,
    )

    current_dataloader = DataLoader(current_dataset, batch_size=32, shuffle=False)

    with torch.no_grad():
        for batch_idx, (inputs, targets, i_list, j_list,) in enumerate(current_dataloader):
            inputs = inputs.to(device)
            outputs = model(inputs)
            outputs_np = outputs.squeeze(1).cpu().numpy()

            denormalized_outputs = []
            for output_data in outputs_np:
                denormalized_data = val_normalizer.inverse_transform(output_data, 'output')
                denormalized_outputs.append(denormalized_data)
            denormalized_outputs = np.array(denormalized_outputs)

            for sample_idx in range(len(i_list)):
                i = i_list[sample_idx].item()
                j = j_list[sample_idx].item()

                output_folder = os.path.join(output_directory, f'i{i}_j{j}')
                os.makedirs(output_folder, exist_ok=True)

                output_file_path = os.path.join(output_folder, f'{input_time}.npy')
                np.save(output_file_path, denormalized_outputs[sample_idx])

    print(f"Time point {target_time} completed.")

print("\nAll time points prediction complete!")